# Attack a password check

#### Learning goals:
- Learn how a "bad" password check looks like
- Learn how to run code from C files on ChipWhisperer
- Learn how to read output from ChipWhisperer
- Learn how to exploit different program flows
- Learn how a SAD attack works

In [1]:
%load_ext autoreload
%autoreload 2

import os
from pathlib import Path

import plotly.graph_objects as pgo
from cwtoolbox import CaptureDevice

In [2]:
# Create a capture device, compile and flash target application

device = CaptureDevice.create("CWLITEXMEGA")
device.compile(file=str(Path("passwordcheck.c").resolve()))
device.flash()

c:\work\securecoding_ws2526\.venv\Lib\site-packages\chipwhisperer\capture\trace\TraceWhisperer.py:31: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources # type: ignore


XMEGA Programming flash...
XMEGA Reading flash...
Verified flash OK, 2239 bytes


In [3]:
# Capture one trace with input b"helloinfineon"

def input_data(index):
    return b"helloinfineon"

trace = device.capture(
    number_of_traces=1,
    input=input_data,
    number_of_samples=1000
)

100%|██████████| 1/1 [00:00<00:00, 45.56it/s]


In [4]:
# Plot the first captured trace
fig = pgo.Figure()
fig.add_trace(pgo.Scatter(y=trace["trace"][0]))
fig.show()

In [5]:
# Capture traces with different inputs

def input_data(index):
    data = [
        b"helloinfineon",
        b"ifx-hello    ",
        b"in...........",
        b"inabcdefghijk",
        b"inf..........",
    ]
    return data[index]

trace = device.capture(
    number_of_traces=5,
    input=input_data,
    number_of_samples=1000
)

100%|██████████| 5/5 [00:00<00:00, 49.80it/s]


In [6]:
# Plot all recorded traces

fig = pgo.Figure()
for i in range(len(trace)):
    fig.add_trace(pgo.Scatter(y=trace["trace"][i], name=str(bytes(trace["input"][i]))))
fig.update_layout(title="Recorded Traces", xaxis_title="Sample Index", yaxis_title="Amplitude")
fig.show()

In [7]:
# Plot all traces and shift each trace by 0.5 down

fig = pgo.Figure()
for i in range(len(trace)):
    fig.add_trace(pgo.Scatter(y=trace["trace"][i] - i * 0.5, name=str(bytes(trace["input"][i]))))
fig.update_layout(title="Recorded Traces", xaxis_title="Sample Index", yaxis_title="Amplitude")
fig.show()

We see that with each additional matching character the significant spot moves to the right.

In [8]:
def sad(trace1, trace2) -> float:
    """Sum of absolute differences"""
    return sum(abs(trace1 - trace2))

In [9]:
# Plot the differences in the traces

# Use first recorded trace as bases
# base_trace = {"trace": [...], "input": [...]}
base_trace = trace[2]

fig = pgo.Figure()
for i in range(len(trace)):
    fig.add_trace(
        pgo.Scatter(
            # Plot the absolute difference of two traces
            y=abs(base_trace["trace"] - trace["trace"][i]) - i * 0.5,
            name=str(bytes(base_trace["input"]))
            + " - "
            + str(bytes(trace["input"][i]))
            + " sad = " + str(sad(base_trace["trace"], trace["trace"][i]))
        )
    )
fig.update_layout(
    title="Recorded Traces", xaxis_title="Sample Index", yaxis_title="Amplitude"
)
fig.show()

We recognize:
- The SAD value is "big" if two traces differ in the amount of correct characters.
- The SAD value is "small" if two traces are equal in the amount of correct characters independent of the other characters.

In [10]:
# Attack using the SAD value
# Assume:
# - The password consists only of string.ascii_lowercase
# - The password is 8 characters long
# - Use sad threshold from values above

import string


def record_attempt(attempt: str):
    return device.capture(
        number_of_traces=1,
        number_of_samples=1000,
        input=lambda _: attempt.encode(),
    )["trace"][0]


def sad_attack(
    characters=string.ascii_lowercase,
    threshold=30,
    password_length=8,
    wrong_character=" ",
):
    # 1. Record base_trace. All characters wrong!
    # 2. Iterate over characters
    # 2a. Construct new guess by appending current character to
    #     already revealed part.
    # 2b. If sad between guess trace and base_trace is bigger than threshold:
    #     add current character to revealed part.
    # 3. Record new base_trace with already revealed password and wrong character.

    already_revealed = ""

    base_trace = record_attempt(password_length * wrong_character)

    for _ in range(password_length):
        for character in characters:
            current_guess = already_revealed + character
            guess_trace = record_attempt(current_guess)
            print(
                "already_revealed:",
                already_revealed,
                "current_guess:",
                current_guess,
                "sad:",
                sad(base_trace, guess_trace),
            )
            if sad(base_trace, guess_trace) > threshold:
                print("\nNEW REVEALED CHARACTER ", character, "\n")
                already_revealed += character
                base_trace = record_attempt(already_revealed + wrong_character)
                break

    return already_revealed


sad_attack()

100%|██████████| 1/1 [00:00<00:00, 47.91it/s]


already_revealed:  current_guess: a sad: 6.5888671875


100%|██████████| 1/1 [00:00<00:00, 48.40it/s]


already_revealed:  current_guess: b sad: 5.8671875


100%|██████████| 1/1 [00:00<00:00, 55.37it/s]


...

already_revealed: infineo current_guess: infineom sad: 4.267578125


100%|██████████| 1/1 [00:00<00:00, 62.35it/s]


already_revealed: infineo current_guess: infineon sad: 39.94140625

NEW REVEALED CHARACTER  n 



100%|██████████| 1/1 [00:00<00:00, 34.30it/s]


'infineon'